# Modeling

We predict the price of the next game a user reviews. Every review after a user's first is a target (1,252,608 of 2,805,451 reviews), and every feature comes from days strictly before it. We fit one gradient boosted tree model (GBM) per metric, choose settings on validation, and score test once on the [Evaluation](evaluation.ipynb) page.

In [1]:
import datetime as dt
from pathlib import Path

import hvplot.polars
import numpy as np
import polars as pl
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from utils import load_games

pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
DATA = Path("data")
SEED = 0

In [2]:
TOP_G = ["Indie", "Action", "Adventure", "Casual", "Strategy", "Simulation", "RPG", "Free to Play",
         "Early Access", "Sports", "Massively Multiplayer", "Racing"]
GEN = [f"g_{g.lower().replace(' ', '_')}" for g in TOP_G]

games = load_games(DATA / "steam_games.json")
meta = games.drop_nulls("price").select(
    "product_id", "price", "release",
    *[pl.col("genres").list.contains(g).cast(pl.Float64).alias(c) for g, c in zip(TOP_G, GEN)])

ev = (pl.read_parquet(DATA / "cache" / "reviews.parquet")
        .select("user_id", "product_id", "date").unique()
        .drop_nulls("user_id")
        .join(meta, on="product_id")
        .sort("user_id", "date", "product_id")
        .unique(["user_id", "product_id"], keep="first", maintain_order=True)
        .with_columns(day=pl.col("date").dt.date(), lp=pl.col("price").log1p(),
                      free=(pl.col("price") == 0).cast(pl.Float64),
                      age=(pl.col("date").dt.date() - pl.col("release")).dt.total_days().clip(-365, 5000) / 365))

T1, T2 = ev["date"].quantile(0.8), ev["date"].quantile(0.9)
SPLITS = ["train", "val", "test"]
ev = ev.with_columns(split=pl.when(pl.col("date") < T1).then(pl.lit("train"))
                             .when(pl.col("date") < T2).then(pl.lit("val")).otherwise(pl.lit("test")))
print(f"{ev.height:,} reviews: train < {T1.date()} <= val < {T2.date()} <= test")

2,805,451 reviews: train < 2017-07-07 <= val < 2017-11-06 <= test


---

## Features

The history features describe what a user reviewed before a given day. We keep running totals per user (count, sum of prices, sum of log prices, share of free games and so on), take the last value of each day and shift it down one row per user, so each review sees the state after the user's previous review. After the shift, a review is not part of its own history, and neither are the other reviews written the same day.

In [3]:
u = pl.col("user_id")
ev = ev.with_columns(ipop=pl.int_range(pl.len()).over("product_id", order_by="date").log1p())

SUMS = ["lp", "price", "free", "ipop", *GEN]
running = ev.with_columns(
    n=pl.int_range(1, pl.len() + 1).over(u),
    n_age=pl.col("age").is_not_null().cum_sum().over(u),
    s_age=pl.col("age").fill_null(0).cum_sum().over(u),
    s_lp2=(pl.col("lp") ** 2).cum_sum().over(u),
    mx=pl.col("price").cum_max().over(u),
    med10=pl.col("price").rolling_median(10, min_samples=1).over(u),
    **{f"ewm{a}": pl.col("price").ewm_mean(alpha=a).over(u) for a in (0.1, 0.3, 0.5, 0.7, 0.9)},
    ewm_lp=pl.col("lp").ewm_mean(alpha=0.1).over(u),
    first_day=pl.col("day").first().over(u),
    **{f"s_{c}": pl.col(c).cum_sum().over(u) for c in SUMS},
)

In [4]:
STATE = ["n", "n_age", "s_age", "s_lp2", "mx", "med10", "ewm_lp", "first_day",
         *[c for c in running.columns if c.startswith("ewm0")], *[f"s_{c}" for c in SUMS]]
LAST = ["last_p", "last_lp", "last_item"]
before = (running.group_by("user_id", "day", maintain_order=True)
                 .agg(pl.col(STATE).last(), last_p=pl.col("price").mean(), last_lp=pl.col("lp").mean(),
                      last_item=pl.col("product_id").last())
                 .with_columns(pl.col(STATE + LAST).shift(1).over(u), prev_day=pl.col("day").shift(1).over(u)))

n = pl.col("n")
X = (ev.select("user_id", "product_id", "day", "split", "price", "lp")
       .join(before, on=["user_id", "day"])
       .filter(n.is_not_null())
       .with_columns(logn=n.log(), m_p=pl.col("s_price") / n, m_lp=pl.col("s_lp") / n,
                     free_sh=pl.col("s_free") / n,
                     sd_lp=(pl.col("s_lp2") / n - (pl.col("s_lp") / n) ** 2).clip(0, None).sqrt(),
                     m_ipop=pl.col("s_ipop") / n, m_age=pl.col("s_age") / pl.col("n_age"),
                     gap=(pl.col("day") - pl.col("prev_day")).dt.total_days().log1p(),
                     tenure=(pl.col("day") - pl.col("first_day")).dt.total_days().log1p(),
                     month=pl.col("day").dt.month(), dow=pl.col("day").dt.weekday(),
                     acct_num=u.cast(pl.Int64) - 76561197960265728,
                     **{f"sh_{c}": pl.col(f"s_{c}") / n for c in GEN}))
X.height

1252608

Here is one user with five targets, one per day. `price` is the target. The other columns are what the model sees: `n` earlier reviews, their mean price `m_p`, the last price `last_p`, and a recency-weighted mean `ewm0.1` where each older review counts 0.9 times as much as the one after it. On 2015-12-02 the user has two earlier reviews, at \$29.99 and \$19.99, so `m_p` is \$24.99.

In [5]:
one = (X.group_by("user_id").agg(targets=pl.len(), days=pl.col("day").n_unique())
        .filter((pl.col("targets") == 5) & (pl.col("days") == 5)).sort("user_id")["user_id"][0])
X.filter(u == one).select("day", "price", "n", "m_p", "last_p", "ewm0.1").sort("day")

day,price,n,m_p,last_p,ewm0.1
date,f64,i64,f64,f64,f64
2015-11-14,19.990,1,29.990,29.990,29.990
2015-12-02,14.990,2,24.990,19.990,24.727
2015-12-10,17.990,3,21.657,14.990,21.134
2016-11-26,34.990,4,20.740,17.990,20.220
2017-10-25,29.990,5,23.590,34.990,23.827


To check the shift, we recompute the history for a random sample of targets the slow way, from reviews strictly before the target day, and compare.

In [6]:
check = X.sample(2000, seed=SEED).select("user_id", "day", "n", "m_lp").unique(["user_id", "day"])
direct = (check.join(ev.select("user_id", pl.col("day").alias("d"), "lp"), on="user_id")
               .filter(pl.col("d") < pl.col("day"))
               .group_by("user_id", "day").agg(n_direct=pl.len(), m_lp_direct=pl.col("lp").mean()))
c = check.join(direct, on=["user_id", "day"])
assert c.height == check.height
assert (c["n"] == c["n_direct"]).all() and np.allclose(c["m_lp"], c["m_lp_direct"])
print("history features match a direct strictly-earlier recomputation on", c.height, "targets")

history features match a direct strictly-earlier recomputation on 1999 targets


Two more groups use other users. The global features are the average price, average log price and free share of all reviews in the last 7 and 30 days, up to yesterday. The Markov features look at the last game a user reviewed and take the average price and free share of the games people reviewed next after it, using only earlier days. We pull these averages toward the training average as if each game had 20 extra transitions at that average, so a game with few followers does not give a noisy number.

In [7]:
gd = (ev.group_by("day").agg(cnt=pl.len(), sp=pl.col("price").sum(), slp=pl.col("lp").sum(), sf=pl.col("free").sum())
        .sort("day").upsample("day", every="1d").fill_null(0))
for w in (7, 30):
    roll = lambda c: pl.col(c).rolling_sum(w, min_samples=1)
    gd = gd.with_columns(**{f"glob_p{w}": (roll("sp") / roll("cnt")).shift(1),
                            f"glob_lp{w}": (roll("slp") / roll("cnt")).shift(1),
                            f"glob_free{w}": (roll("sf") / roll("cnt")).shift(1)})
X = X.join(gd.drop("cnt", "sp", "slp", "sf"), on="day", how="left")

K = 20
train_rows = X.filter(pl.col("split") == "train")
PRIOR = {"lp": train_rows["lp"].mean(), "p": train_rows["price"].mean(), "f": (train_rows["price"] == 0).mean()}
trans = (X.group_by("last_item", "day")
          .agg(t_n=pl.len(), t_lp=pl.col("lp").sum(), t_p=pl.col("price").sum(), t_f=(pl.col("price") == 0).sum())
          .sort("last_item", "day")
          .with_columns(pl.col("t_n", "t_lp", "t_p", "t_f").cum_sum().shift(1).over("last_item").fill_null(0)))
X = (X.join(trans, on=["last_item", "day"], how="left")
      .with_columns(tr_lp=(pl.col("t_lp") + K * PRIOR["lp"]) / (pl.col("t_n") + K),
                    tr_p=(pl.col("t_p") + K * PRIOR["p"]) / (pl.col("t_n") + K),
                    tr_f=(pl.col("t_f") + K * PRIOR["f"]) / (pl.col("t_n") + K),
                    tr_logn=pl.col("t_n").log1p()))
X.height

1252608

The last group is the Steam calendar. From yesterday's reviews we take the price of the most reviewed game, its share of that day's reviews and the review-weighted price of the top 10. The sale dates, and the features made from them, are described below.

In [8]:
yesterday = (ev.group_by("day", "product_id").agg(reviews=pl.len(), price=pl.col("price").first())
               .sort("day", "reviews", "product_id", descending=[False, True, False])
               .group_by("day", maintain_order=True)
               .agg(top1_p=pl.col("price").first(),
                    top1_share=pl.col("reviews").first() / pl.col("reviews").sum(),
                    top10_p=(pl.col("price") * pl.col("reviews")).head(10).sum() / pl.col("reviews").head(10).sum())
               .with_columns(day=pl.col("day") + pl.duration(days=1)))
yesterday.height

2612

The sale calendar covers Steam's three big sales each year: summer (June 22 to July 5), autumn (the day before Thanksgiving to five days after) and winter (December 21 to January 4).

In [9]:
def thanksgiving(year):
    first = dt.date(year, 11, 1)
    return first + dt.timedelta(days=(3 - first.weekday()) % 7 + 21)


def sales(year):
    tg = thanksgiving(year)
    return [(1, dt.date(year, 6, 22), dt.date(year, 7, 5)),
            (2, tg - dt.timedelta(days=1), tg + dt.timedelta(days=5)),
            (3, dt.date(year, 12, 21), dt.date(year + 1, 1, 4))]

For each day we record which sale is running (if any), the day inside it, the days to the next sale and since the last one, and the days to the nearest summer, autumn and winter sale.

In [10]:
def sale_calendar(days):
    windows = [w for y in range(days[0].year - 1, days[-1].year + 2) for w in sales(y)]
    rows = []
    for d in days:
        inside = [(k, s) for k, s, e in windows if s <= d <= e]
        kind, start = inside[0] if inside else (0, None)
        nearest = {k: min(((d - s).days for kk, s, e in windows if kk == k), key=abs) for k in (1, 2, 3)}
        rows.append({"day": d, "sale_kind": kind, "sale_day": (d - start).days if inside else -1,
                     "sale_to_next": 0 if inside else min((s - d).days for k, s, e in windows if s > d),
                     "sale_since": 0 if inside else min((d - e).days for k, s, e in windows if e < d),
                     "d_summer": nearest[1], "d_autumn": nearest[2], "d_winter": nearest[3]})
    return pl.DataFrame(rows).with_columns(pl.exclude("day").cast(pl.Float64))


calendar = sale_calendar(X["day"].unique().sort().to_list())
X = X.join(yesterday, on="day", how="left").join(calendar, on="day", how="left")
X.height

1252608

We check the trending features the same way. For 150 random days, we rebuild yesterday's top games directly from the reviews and compare them with the table.

In [11]:
days = X["day"].unique().sort().sample(150, seed=SEED)
rows = []
for d in days:
    top = (ev.filter(pl.col("day") == d - dt.timedelta(days=1)).group_by("product_id")
             .agg(c=pl.len(), price=pl.col("price").first()).sort("c", "product_id", descending=[True, False]))
    if top.height:
        rows.append({"day": d, "top1_direct": top["price"][0],
                     "top10_direct": (top["price"] * top["c"]).head(10).sum() / top["c"].head(10).sum()})
direct = pl.DataFrame(rows)
c = direct.join(X.unique("day").select("day", "top1_p", "top10_p"), on="day")
assert np.allclose(c["top1_direct"], c["top1_p"]) and np.allclose(c["top10_direct"], c["top10_p"])
print("yesterday's top games match a direct recomputation on", c.height, "days")

yesterday's top games match a direct recomputation on 150 days


In total there are 50 features.

| group | features | what it holds |
|---|---|---|
| history | 11 | count, mean, max, median of the last 10 and spread of past prices, share of free games, last price, recency-weighted prices |
| taste | 14 | mean age and popularity of past games, share of each of 12 genres |
| timing | 4 | days since the last review, days since the first, month, weekday |
| Markov | 4 | price and free share of the games reviewed after the last game, and how many transitions that rests on |
| global | 6 | average price, log price and free share of all reviews, last 7 and 30 days |
| account | 1 | account number |
| trending and sales | 10 | price of yesterday's top games, position in the Steam sale calendar |

In [12]:
HIST = ["logn", "m_p", "m_lp", "free_sh", "sd_lp", "mx", "med10", "ewm0.1", "ewm_lp", "last_p", "last_lp"]
TASTE = ["m_ipop", "m_age", *[f"sh_{c}" for c in GEN]]
TIMING = ["gap", "tenure", "month", "dow"]
MARKOV = ["tr_lp", "tr_p", "tr_f", "tr_logn"]
GLOBAL = ["glob_p7", "glob_lp7", "glob_free7", "glob_p30", "glob_lp30", "glob_free30"]
ACCOUNT = ["acct_num"]
CONTEXT = ["top1_p", "top1_share", "top10_p", "sale_kind", "sale_day", "sale_to_next", "sale_since",
           "d_summer", "d_autumn", "d_winter"]
FEATURES = HIST + TASTE + TIMING + MARKOV + GLOBAL + ACCOUNT + CONTEXT

NAMES = {"logn": "Log number of past reviews", "m_p": "Mean past price", "m_lp": "Mean past log price",
         "free_sh": "Share of free games", "sd_lp": "Spread of log price", "mx": "Max past price",
         "med10": "Median price, last 10", "ewm0.1": "Recency-weighted price", "ewm_lp": "Recency-weighted log price",
         "last_p": "Last review's price", "last_lp": "Previous day's log price",
         "m_ipop": "Mean popularity of past games", "m_age": "Mean age of past games",
         "gap": "Days since previous review", "tenure": "Days since first review", "month": "Month", "dow": "Weekday",
         "tr_lp": "Next-game log price after last game", "tr_p": "Next-game price after last game",
         "tr_f": "Next-game free share after last game", "tr_logn": "Transitions seen from last game",
         "glob_p7": "Mean price, last 7 days", "glob_lp7": "Mean log price, last 7 days",
         "glob_free7": "Free share, last 7 days", "glob_p30": "Mean price, last 30 days",
         "glob_lp30": "Mean log price, last 30 days", "glob_free30": "Free share, last 30 days",
         "acct_num": "Account number", "top1_p": "Price of yesterday's top game",
         "top1_share": "Share of yesterday's top game", "top10_p": "Mean price of yesterday's top 10",
         "sale_kind": "Sale type", "sale_day": "Day of sale", "sale_to_next": "Days to next sale",
         "sale_since": "Days since last sale", "d_summer": "Days from summer sale",
         "d_autumn": "Days from autumn sale", "d_winter": "Days from winter sale",
         **{f"sh_{c}": f"Share of {g} games" for g, c in zip(TOP_G, GEN)}}
print(len(FEATURES), "features")

50 features


---

## Models

The best guess depends on the loss. Under squared error on dollars it is the mean, under absolute error it is the median, and under squared error on log price it is the geometric mean. So we fit three heads: `log` (squared loss on log price, for MSLE), `abs` (absolute loss on dollars, for MAE) and `sq` (squared loss on dollars, for MSE). All three are scikit-learn's `HistGradientBoostingRegressor`.

The table compares the approaches we tried. The constant, last-price and Markov rows are the simple baselines from the introduction. Ridge and linear regression are the linear comparisons. The GBMs are the main model, and the hurdle model is a variant of them that we tried.

| approach | assumes | cost to fit | how to read it | where it helps or fails here |
|---|---|---|---|---|
| constant guess | prices don't depend on the user | none | one number | the floor every model has to beat |
| last price, or recent mean | the next price is close to recent prices | none | very direct | noisy on its own, since one unusual game moves it |
| Markov chain on price buckets or games | the next price depends on the last price bucket or game | a lookup table | an average per state | captures regression to the mean, but rare games have few counts, so we shrink toward the average |
| ridge | the log price is linear in the features | one linear solve | coefficients | cannot model a threshold such as "free" |
| linear regression on history | the log price is linear in the history features | one linear solve | coefficients | a baseline on the Evaluation page |
| gradient boosted trees (main model) | the price depends on the features in nonlinear ways | one tree ensemble per head | low, so we use permutation importance | handles interactions and mixed features, but cannot extrapolate beyond the prices it trained on |
| hurdle model (variant) | free or paid is a separate decision | a classifier and two regressors | a probability times a paid price | no measurable difference from the plain GBM on test |

We use scikit-learn's `HistGradientBoostingRegressor` rather than XGBoost or LightGBM. It handles missing values and sits in the same library as everything else we use. We did not compare it with XGBoost or LightGBM.

We fix `min_samples_leaf` at 500 (not tuned), so no leaf rests on fewer than 500 reviews and one unusual game cannot drive a split.

Predictions below zero are clipped to zero, since a price can't be negative.

In [13]:
def scores(y, p):
    p = np.clip(p, 0, None)
    return {"MAE": np.mean(np.abs(y - p)), "MSE": np.mean((y - p) ** 2),
            "MSLE": np.mean((np.log1p(y) - np.log1p(p)) ** 2)}

The log head trains on `log1p(price)` and converts its predictions back to dollars with `expm1`. The other two heads train on dollars.

In [14]:
HEADS = {"log": dict(loss="squared_error", target="log", metric="MSLE"),
         "abs": dict(loss="absolute_error", target="raw", metric="MAE"),
         "sq": dict(loss="squared_error", target="raw", metric="MSE")}


def fit_head(head, A, y, **params):
    spec = HEADS[head]
    return HistGradientBoostingRegressor(loss=spec["loss"], random_state=SEED, **params).fit(
        A, np.log1p(y) if spec["target"] == "log" else y)


def predict_head(head, m, B):
    p = m.predict(B)
    return np.expm1(p) if HEADS[head]["target"] == "log" else p

The three date splits:

In [15]:
tr, va, te = (X.filter(pl.col("split") == s) for s in SPLITS)
y_tr, y_va, y_te = (d["price"].to_numpy() for d in (tr, va, te))
print(f"Targets: {tr.height:,} train, {va.height:,} validation, {te.height:,} test")
print(f"Free share of train targets: {(y_tr == 0).mean():.1%}")

Targets: 959,337 train, 150,066 validation, 143,205 test
Free share of train targets: 16.4%


First we check which feature groups earn their place. We add them one at a time on a 250,000-row sample of train with 200 trees, and score on validation. Every group lowers MSLE and MSE. The Markov features give the biggest step after the history features (MSLE 1.481 to 1.463, MSE 281.9 to 276.8). Account number lowers MSLE and MSE but raises MAE a little (11.703 to 11.720). Trending and sales help all three. Each row comes from one sample and one seed. In repeated runs, scores moved by about 0.001 MSLE, 0.4 MSE and 0.02 MAE from seed to seed, so steps smaller than that, such as account number's +0.017 MAE, are within noise.

In [16]:
sample = tr.sample(250_000, seed=SEED)
y_s = sample["price"].to_numpy()
GROUPS = {"history": HIST, "+ taste": HIST + TASTE, "+ timing": HIST + TASTE + TIMING,
          "+ Markov": HIST + TASTE + TIMING + MARKOV, "+ global": HIST + TASTE + TIMING + MARKOV + GLOBAL,
          "+ account number": HIST + TASTE + TIMING + MARKOV + GLOBAL + ACCOUNT, "+ trending and sales": FEATURES}
rows = []
for name, cols in GROUPS.items():
    A, B = sample.select(cols).to_numpy(), va.select(cols).to_numpy()
    row = {"features": name}
    for head, spec in HEADS.items():
        m = fit_head(head, A, y_s, max_iter=200)
        row[spec["metric"]] = scores(y_va, predict_head(head, m, B))[spec["metric"]]
    rows.append(row)
pl.DataFrame(rows).select("features", "MSLE", "MAE", "MSE")

features,MSLE,MAE,MSE
str,f64,f64,f64
"""history""",1.502,11.866,284.743
"""+ taste""",1.488,11.834,283.783
"""+ timing""",1.481,11.759,281.936
"""+ Markov""",1.463,11.712,276.781
"""+ global""",1.457,11.703,274.522
"""+ account number""",1.450,11.720,273.207
"""+ trending and sales""",1.445,11.642,271.316


Next we try four settings per head, a learning rate of 0.05 or 0.1 with 31 or 127 leaves per tree, on the same sample with 400 trees. The settings change each score very little. Across the four settings, the best and the worst differ by 0.002 MSLE, 0.02 MAE and 0.6 MSE. We take the best per head.

In [17]:
GRID = [dict(learning_rate=lr, max_leaf_nodes=leaves, min_samples_leaf=500, max_iter=400)
        for lr in (0.05, 0.1) for leaves in (31, 127)]
A, B = sample.select(FEATURES).to_numpy(), va.select(FEATURES).to_numpy()
rows = []
for head, spec in HEADS.items():
    for i, params in enumerate(GRID):
        m = fit_head(head, A, y_s, **params)
        rows.append({"head": head, "metric": spec["metric"], "config": i, "learning_rate": params["learning_rate"],
                     "leaves": params["max_leaf_nodes"], "score": scores(y_va, predict_head(head, m, B))[spec["metric"]]})
tuning = pl.DataFrame(rows)
BEST = {head: GRID[tuning.filter(pl.col("head") == head).sort("score")["config"][0]] for head in HEADS}
tuning.drop("config")

head,metric,learning_rate,leaves,score
str,str,f64,i64,f64
"""log""","""MSLE""",0.050,31,1.441
"""log""","""MSLE""",0.050,127,1.441
"""log""","""MSLE""",0.100,31,1.442
"""log""","""MSLE""",0.100,127,1.443
"""abs""","""MAE""",0.050,31,11.628
"""abs""","""MAE""",0.050,127,11.621
"""abs""","""MAE""",0.100,31,11.637
"""abs""","""MAE""",0.100,127,11.620
"""sq""","""MSE""",0.050,31,269.997


The next price piles up at a few values: \$19.99 is the most common (18.8% of train targets), then free (16.4%). One regressor has to cover both, which pulls its predictions toward the middle, so we also tried a hurdle model. A classifier gives the chance `p` that the next game is free, and regressors predict the price of paid games. For MSE the prediction is `(1 - p)` times the paid price, and for MSLE it is `expm1((1 - p) * E[log1p(price) | paid])`. We average it with the plain GBM (in log space for MSLE). MAE keeps the plain head, since the median of a mix of free and paid games isn't `(1 - p)` times the paid price. The hurdle models use default settings and were not tuned.

In [18]:
price_share = (tr.group_by("price").agg(n=pl.len()).sort("n", descending=True).head(8)
                 .with_columns(percent=(100 * pl.col("n") / tr.height).round(1)).drop("n"))
price_share

price,percent
f64,f64
19.990,18.800
0.000,16.400
9.990,14.300
14.990,11.200
29.990,7.400
39.990,5.700
4.990,5.200
24.990,3.300


In [19]:
GBM = dict(max_iter=300, learning_rate=0.1, random_state=SEED)


def fit_hurdle(A, y):
    paid = y > 0
    return {"free": HistGradientBoostingClassifier(**GBM).fit(A, ~paid),
            "dollars": HistGradientBoostingRegressor(**GBM).fit(A[paid], y[paid]),
            "log": HistGradientBoostingRegressor(**GBM).fit(A[paid], np.log1p(y[paid]))}


def predict_hurdle(m, B):
    p_free = m["free"].predict_proba(B)[:, 1]
    return {"sq": (1 - p_free) * m["dollars"].predict(B), "log": np.expm1((1 - p_free) * m["log"].predict(B))}


def blend(head, a, b):
    a, b = np.clip(a, 0, None), np.clip(b, 0, None)
    return np.expm1((np.log1p(a) + np.log1p(b)) / 2) if head == "log" else (a + b) / 2

We fit the three plain heads, the hurdle variants and a ridge regression (standardized features, log target) on the same features, and time each fit.

In [20]:
import time

TIMES = {}


def fit_all(d):
    A, y = d.select(FEATURES).to_numpy(), d["price"].to_numpy()
    plain = {}
    for head in HEADS:
        t0 = time.perf_counter()
        plain[head] = fit_head(head, A, y, **BEST[head])
        TIMES[f"GBM, {HEADS[head]['metric']} head"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    hurdle = fit_hurdle(A, y)
    TIMES["hurdle: classifier and two regressors"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(np.nan_to_num(A), np.log1p(y))
    TIMES["ridge"] = time.perf_counter() - t0
    return plain, hurdle, ridge

In [21]:
def predict_all(models, d):
    plain, hurdle, ridge = models
    B = d.select(FEATURES).to_numpy()
    out = {f"plain_{head}": predict_head(head, m, B) for head, m in plain.items()}
    pred_hurdle = predict_hurdle(hurdle, B)
    out |= {f"hurdle_{head}": p for head, p in pred_hurdle.items()}
    out |= {f"blend_{head}": blend(head, out[f"plain_{head}"], pred_hurdle[head]) for head in ("log", "sq")}
    out["ridge"] = np.expm1(ridge.predict(np.nan_to_num(B)))
    return out

Everything is fit on train and scored on validation, the only data we choose on. For each metric we pick the lowest validation score.

The hurdle blends edge out the plain heads by 0.003 MSLE and 0.2 MSE. That margin disappears on test (see [Evaluation](evaluation.ipynb)), so in practice the plain GBMs carry the result. Ridge is clearly worse: MSLE 1.472 against 1.436, and MAE 12.53 against 11.59.

The course models are the reference points. On validation, the shrunk user mean (the shrinkage model from homework 2) reaches MSLE 1.520, the linear regression on history 1.534 and the Markov chain on games 1.551 (see [Evaluation](evaluation.ipynb)). The GBM uses the Markov chain's output as one of its features, and gets to 1.436 by combining it with everything else.

In [22]:
val_models = fit_all(tr)
val_pred = predict_all(val_models, va)

CHOICES = {"MSLE": ["plain_log", "hurdle_log", "blend_log"], "MSE": ["plain_sq", "hurdle_sq", "blend_sq"],
           "MAE": ["plain_abs"]}
CANDIDATES = ["plain_abs", "plain_log", "plain_sq", "hurdle_log", "hurdle_sq", "blend_log", "blend_sq", "ridge"]
PICK = {m: min(ks, key=lambda k: scores(y_va, val_pred[k])[m]) for m, ks in CHOICES.items()}
val_table = pl.DataFrame([{"model": k, **scores(y_va, val_pred[k]),
                           "picked for": ", ".join(m for m in PICK if PICK[m] == k)} for k in CANDIDATES])
val_table

model,MAE,MSE,MSLE,picked for
str,f64,f64,f64,str
"""plain_abs""",11.589,283.620,1.562,"""MAE"""
"""plain_log""",12.344,332.147,1.439,""""""
"""plain_sq""",11.846,269.114,1.736,""""""
"""hurdle_log""",12.379,334.462,1.439,""""""
"""hurdle_sq""",11.818,269.654,1.717,""""""
"""blend_log""",12.356,333.178,1.436,"""MSLE"""
"""blend_sq""",11.820,268.924,1.726,"""MSE"""
"""ridge""",12.529,337.665,1.472,""""""


With the choices fixed, we refit every model on train plus validation and predict test. Test reviews are never training rows. A test review does serve as history for a later review by the same user, which is what Steam would know on that day. The timing table below shows how long each model took in this refit: the GBM heads take 28 to 66 seconds each, the hurdle models 43 seconds, and ridge under one second. We save the predictions so the Evaluation page can score them against the baselines.

In [23]:
test_models = fit_all(pl.concat([tr, va]))
test_pred = predict_all(test_models, te)
print("Picks:", PICK)
timing = pl.DataFrame({"model": list(TIMES), "seconds": [round(v, 1) for v in TIMES.values()]})
timing

Picks: {'MSLE': 'blend_log', 'MSE': 'blend_sq', 'MAE': 'plain_abs'}


model,seconds
str,f64
"""GBM, MSLE head""",62.000
"""GBM, MAE head""",66.400
"""GBM, MSE head""",28.000
"""hurdle: classifier and two reg…",43.400
"""ridge""",0.800


In [24]:
KEEP = ["user_id", "day", "split", "price", "lp", "n", *HIST, "ewm0.3", "ewm0.5", "ewm0.7", "ewm0.9", "tr_lp"]


def with_predictions(d, pred):
    return d.select(KEEP).with_columns(**{f"p_{k}": pl.Series(v) for k, v in pred.items()},
                                       **{f"p_ours_{m}": pl.Series(pred[k]) for m, k in PICK.items()})


(pl.concat([tr.select(KEEP), with_predictions(va, val_pred), with_predictions(te, test_pred)], how="diagonal")
   .write_parquet(DATA / "cache" / "final_eval.parquet"))

---

## What the model uses

To see which features the model leans on, we shuffle one column of validation at a time and measure how much MSLE rises, using the log head fit on train. The recency-weighted log price is first (+0.038), then account number (+0.033), the number of past reviews (+0.027) and the log price of what people review after the last game (+0.018). Steam IDs are handed out in order, so a bigger account number usually means a newer account. We do not have account creation dates to check that, so read the feature as a rough stand-in for age.

In [25]:
rows = va.sample(30_000, seed=SEED)


def msle_scorer(est, A, y):
    return -np.mean((np.log1p(y) - np.log1p(np.clip(np.expm1(est.predict(A)), 0, None))) ** 2)


pi = permutation_importance(val_models[0]["log"], rows.select(FEATURES).to_numpy(), rows["price"].to_numpy(),
                            scoring=msle_scorer, n_repeats=3, random_state=SEED)
imp = (pl.DataFrame({"feature": FEATURES, "MSLE_increase": pi.importances_mean})
         .with_columns(pl.col("feature").replace(NAMES)).sort("MSLE_increase", descending=True))
imp.head(15).hvplot.barh("feature", "MSLE_increase", width=640, height=400, flip_yaxis=True, color="#4c78a8",
                         xlabel="", ylabel="validation MSLE increase when the feature is shuffled")

:Bars   [feature]   (MSLE_increase)

The table lists the top 15. Each value is the average rise in validation MSLE over three shuffles, so a bigger value means the model relies on that feature more. Features that carry the same information share credit, so a feature can look small even when its group matters.

In [26]:
imp.head(15)

feature,MSLE_increase
str,f64
"""Recency-weighted log price""",0.038
"""Account number""",0.033
"""Log number of past reviews""",0.027
"""Next-game log price after last…",0.018
"""Share of Free to Play games""",0.016
"""Days since previous review""",0.011
"""Next-game price after last gam…",0.011
"""Mean past log price""",0.010
"""Mean past price""",0.006


---

## Ceiling

How low could the error go with better information? We give the model something it cannot have: the genres of the game the user is about to review. MAE drops from 11.64 to 7.99, which is most of the drop we get from adding that information. Adding that game's age and popularity brings it to 6.82. Our features see the next game only indirectly, through the Markov chain features, so this run measures how much of the error comes from not knowing which game is next. These runs use the 250,000-row sample and the absolute head on validation.

In [27]:
target_info = ev.select("user_id", "product_id", *[pl.col(c).alias(f"t_{c}") for c in GEN],
                        t_age=pl.col("age"), t_ipop=pl.col("ipop"))
oracle = X.join(target_info, on=["user_id", "product_id"])
T_GEN = [f"t_{c}" for c in GEN]
o_tr = oracle.filter(pl.col("split") == "train").sample(250_000, seed=SEED)
o_va = oracle.filter(pl.col("split") == "val")
rows = []
for name, cols in {"our features": FEATURES, "+ the next game's genres": FEATURES + T_GEN,
                   "+ its age and popularity": FEATURES + T_GEN + ["t_age", "t_ipop"]}.items():
    m = fit_head("abs", o_tr.select(cols).to_numpy(), o_tr["price"].to_numpy(), max_iter=200)
    rows.append({"features": name, "MAE": scores(o_va["price"].to_numpy(), predict_head("abs", m, o_va.select(cols).to_numpy()))["MAE"]})
pl.DataFrame(rows)

features,MAE
str,f64
"""our features""",11.642
"""+ the next game's genres""",7.985
"""+ its age and popularity""",6.822


---

## Earlier experiments

Before settling on the features above, we tried several other ideas in earlier runs on the full training set. Some of them, with their scores, are in the [earlier notebook](model_evals_edwin.ipynb) under Notebooks. Account number and the features for yesterday's top games and the sale calendar were kept. A ridge model on which games the user reviewed, a set of 20 catalogue tag shares, a richer price history, a fix for prices drifting over time, reviews from accounts with no numeric id, matrix factorization factors (ALS) and TF-IDF of review text did not earn a place, so they are left out. The ablation above, on a 250,000-row sample, is the check that the kept groups still help.

We also left out play time and library size. Both were measured when the data was crawled, not when the review was written, so they would tell the model about the future. For the same reason, the price we use is the list price at crawl time, not what the user paid.